# Week 2 — EDA (sub-team 3)

This notebook is the discussion surface; the numbers and figures come from
`src/data_prep/eda.py`, so they can be regenerated with one command and cannot drift from the data:

```bash
python -m src.data_prep.normalize --split all --write    # train/val/test_cohort.csv
python -m src.data_prep.eda                              # tables + 6 figures
```

Run the two commands first; the cells below only read the outputs.

In [ ]:
import sys
from pathlib import Path

# run from anywhere: walk up until the repository root (the folder holding src/) is found
root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src").exists())
sys.path.insert(0, str(root))

from src.data_prep.eda import (by_brand, by_mileage, by_year, depreciation_by_brand,
                               load, platform_gap, price_overview, summary_stats)
from src.common import config

frame = load(config.DATA_MASTER / "train_cohort.csv")
usable = frame[frame["price_vnd"].notna()]
len(frame), len(usable)

## 0. Bảng thống kê mô tả (descriptive statistics)
Trên cohort mô hình hoá, nên `count` chính là số dòng tuần 3 thực sự train được.

In [ ]:
summary_stats(frame)

## 1. Price distribution per platform
The p90/p10 ratio is the argument for modelling `log(price)`.

In [ ]:
price_overview(usable)

## 2. Median price by brand

In [ ]:
by_brand(usable, top=15)

## 3. Median price by model year

In [ ]:
by_year(usable).tail(15)

## 4. Median price by odometer decile

In [ ]:
by_mileage(usable)

## 5. Depreciation to age 5, by brand
Cross-sectional: different cars at different ages, advertised prices. A market-level description,
not one vehicle's history.

In [ ]:
depreciation_by_brand(usable)

## 6. Matched price gap between platforms
Leave-one-out on brand+model+year, so a platform that only lists newer cars is not rewarded for it.

In [ ]:
platform_gap(usable)

## Figures
Written by `python -m src.data_prep.eda` to `data/master/eda/`: price distribution, price by
brand, price by year, price by odometer, depreciation by brand, platform gap. The same numbers are
in `eda_summary.md` — paste that into the report instead of re-typing them.

In [ ]:
from IPython.display import Image, display

figure_dir = config.DATA_MASTER / "eda"
for name in ("price_distribution", "price_by_brand", "price_by_year",
             "price_by_mileage", "depreciation_by_brand", "platform_gap"):
    path = figure_dir / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path), width=640))
    else:
        print(f"missing {path} — run: python -m src.data_prep.pipeline --steps eda")